# IBM Predictive Maintenance - Interview Walkthrough

This notebook is structured for live interview delivery:

1. Business framing
2. Quick telemetry EDA
3. Model quality and thresholding
4. Operational alert strategy
5. Business recommendations

In [1]:
from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

project_root = Path.cwd().resolve().parent
artifacts_dir = project_root / "artifacts"
processed_data_path = project_root / "data" / "processed" / "synthetic_maintenance.csv"
metrics_path = artifacts_dir / "metrics.json"
feature_importance_path = artifacts_dir / "feature_importance.csv"
scored_path = artifacts_dir / "scored_test.csv"
alerts_path = artifacts_dir / "alerts.csv"

print("Project root:", project_root)

Project root: /Users/danielharrod/Data_Science/IBM_Predictive_Maintenance_ML


In [2]:
# Ensure artifacts exist so notebook is runnable end-to-end.
if not metrics_path.exists() or not scored_path.exists():
    print("Artifacts not found. Running training script...")
    subprocess.run(["python3", str(project_root / "src" / "train.py")], check=True)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

raw_data = pd.read_csv(processed_data_path)
scored = pd.read_csv(scored_path)
alerts = pd.read_csv(alerts_path)
feature_importance = pd.read_csv(feature_importance_path)

print("Rows in telemetry dataset:", len(raw_data))
print("Rows in holdout scored set:", len(scored))
print("Current alert count:", len(alerts))
print("Selected model:", metrics["selected_model"])

Rows in telemetry dataset: 72000
Rows in holdout scored set: 14400
Current alert count: 25
Selected model: logistic_regression


## 1) Problem Framing

Goal: flag assets with elevated failure risk in the next 7 days so operations can intervene early.

Primary interview message:

- This is not just model accuracy.
- This is a decision system balancing missed failures vs alert fatigue.

In [3]:
failure_rate = raw_data["FailureNext7Days"].mean()
print(f"Overall failure-next-7-days rate: {failure_rate:.2%}")

for col in ["EquipmentType", "OperatingMode", "Site"]:
    summary = (
        raw_data.groupby(col, dropna=False)["FailureNext7Days"]
        .agg(["count", "mean"])
        .rename(columns={"count": "n", "mean": "failure_rate"})
        .sort_values("failure_rate", ascending=False)
    )
    print(f"\nFailure rate by {col}:")
    display(summary)

corr_cols = [
    "FailureNext7Days",
    "SensorVibration",
    "SensorTemp",
    "SensorPressure",
    "DaysSinceMaintenance",
    "AssetAgeYears",
    "OperatingHours",
]
corr = raw_data[corr_cols].corr(numeric_only=True)
print("\nCorrelation snapshot to target:")
display(corr[["FailureNext7Days"]].sort_values("FailureNext7Days", ascending=False))

Overall failure-next-7-days rate: 17.86%

Failure rate by EquipmentType:


,n,failure_rate
EquipmentType,,
Turbine,17100,0.263918
Compressor,19080,0.157390
Pump,20340,0.150737
Conveyor,15480,0.147093



Failure rate by OperatingMode:


,n,failure_rate
OperatingMode,,
HighLoad,21589,0.268516
Normal,39771,0.143798
Eco,10640,0.126222



Failure rate by Site:


,n,failure_rate
Site,,
Seattle,17820,0.182772
Chicago,17640,0.182143
Austin,17460,0.176289
Dallas,19080,0.173532



Correlation snapshot to target:


,FailureNext7Days
FailureNext7Days,1.000000
SensorVibration,0.148278
SensorTemp,0.140568
DaysSinceMaintenance,0.125603
AssetAgeYears,0.088272
OperatingHours,0.022297
SensorPressure,0.009165


## 2) Model Performance

The training script compares logistic regression and random forest with CV ROC-AUC, then evaluates holdout performance.

For predictive maintenance, I focus on:

- Ranking quality (ROC-AUC, PR-AUC)
- Operational quality at chosen threshold (precision + recall)

In [4]:
print("Cross-validated ROC-AUC by model:")
for model_name, auc in metrics["cv_roc_auc"].items():
    print(f"  {model_name}: {auc:.3f}")

print("\nSelected model:", metrics["selected_model"])
print("Decision threshold:", metrics["decision_threshold"])

print("\nHoldout test metrics:")
for metric_name, value in metrics["test_metrics"].items():
    print(f"  {metric_name}: {value:.3f}")

print("\nSubgroup checks:")
for group_name, group_values in metrics["subgroup_metrics"].items():
    print(f"\n{group_name}")
    if not group_values:
        print("  (no eligible segments)")
        continue
    display(pd.DataFrame(group_values).T.sort_values("n_samples", ascending=False))

Cross-validated ROC-AUC by model:
  logistic_regression: 0.702
  random_forest: 0.696

Selected model: logistic_regression
Decision threshold: 0.69

Holdout test metrics:
  roc_auc: 0.698
  pr_auc: 0.387
  precision: 0.760
  recall: 0.006
  recall_at_precision_70: 0.006

Subgroup checks:

EquipmentType


,n_samples,precision,recall
Pump,4068.0,0.00,0.000000
Compressor,3816.0,0.00,0.000000
Turbine,3420.0,0.76,0.018482
Conveyor,3096.0,0.00,0.000000



OperatingMode


,n_samples,precision,recall
Normal,7945.0,0.00,0.000000
HighLoad,4244.0,0.76,0.013738
Eco,2211.0,0.00,0.000000


## 3) Threshold Strategy and Alert Operations

A good interview answer includes how to tune threshold based on field capacity.

If maintenance teams can only process a limited number of alerts per week,
choose a threshold that maximizes recall under that capacity constraint.

In [5]:
y_true = scored["actual_failure"].astype(int)
y_prob = scored["failure_probability"].astype(float)

rows = []
for threshold in np.arange(0.20, 0.91, 0.05):
    y_pred = (y_prob >= threshold).astype(int)
    rows.append(
        {
            "threshold": round(float(threshold), 2),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "alert_rate": float(y_pred.mean()),
            "n_alerts": int(y_pred.sum()),
        }
    )

threshold_table = pd.DataFrame(rows)
display(
    threshold_table.style.format(
        {
            "precision": "{:.3f}",
            "recall": "{:.3f}",
            "f1": "{:.3f}",
            "alert_rate": "{:.3f}",
        }
    )
)

print("Top 10 highest-risk alerts:")
display(alerts.sort_values("failure_probability", ascending=False).head(10))

,threshold,precision,recall,f1,alert_rate,n_alerts
0,0.200000,0.330,0.657,0.439,0.431,6203
1,0.250000,0.377,0.513,0.435,0.294,4237
2,0.300000,0.416,0.387,0.401,0.201,2897
3,0.350000,0.450,0.286,0.350,0.137,1977
4,0.400000,0.476,0.192,0.274,0.087,1259
5,0.450000,0.490,0.124,0.198,0.055,785
6,0.500000,0.547,0.079,0.138,0.031,448
7,0.550000,0.571,0.048,0.088,0.018,259
8,0.600000,0.605,0.024,0.046,0.009,124
9,0.650000,0.661,0.012,0.023,0.004,56


Top 10 highest-risk alerts:


,DayIndex,EquipmentType,Site,OperatingMode,AssetAgeYears,OperatingHours,SensorVibration,SensorTemp,SensorPressure,DaysSinceMaintenance,MaintenanceEventToday,actual_failure,failure_probability,predicted_failure,alert
0,145,Turbine,Austin,HighLoad,16,14.447835,3.887252,85.471705,111.827137,53,0,1,0.773387,1,ALERT
1,153,Turbine,Chicago,HighLoad,10,18.641662,4.294100,81.619004,118.761034,59,0,1,0.759935,1,ALERT
2,169,Turbine,Seattle,HighLoad,14,19.024747,3.906038,79.367340,117.945715,58,0,0,0.754285,1,ALERT
3,164,Turbine,Chicago,HighLoad,14,13.373869,4.704758,77.194187,70.854070,50,0,1,0.744604,1,ALERT
4,169,Turbine,Seattle,HighLoad,13,15.457760,4.688088,78.203560,102.782366,46,0,0,0.739039,1,ALERT
5,148,Turbine,Chicago,HighLoad,14,13.409660,4.052764,76.130351,111.172255,61,0,0,0.735866,1,ALERT
6,175,Turbine,Seattle,HighLoad,13,14.231440,4.262197,79.340124,105.378978,52,0,1,0.730757,1,ALERT
7,154,Turbine,Seattle,HighLoad,16,13.643415,4.238042,71.703246,91.489229,56,0,1,0.725343,1,ALERT
8,145,Turbine,Chicago,HighLoad,10,14.305181,4.499770,80.627604,111.943281,51,0,1,0.724375,1,ALERT
9,147,Turbine,Seattle,HighLoad,16,15.475472,3.919721,76.566550,91.757894,52,0,1,0.720751,1,ALERT


In [6]:
print("Top 15 model drivers:")
display(feature_importance.head(15))

Top 15 model drivers:


,feature,importance
0,cat__OperatingMode_Eco,0.497446
1,cat__OperatingMode_Normal,0.469541
2,cat__EquipmentType_Pump,0.439024
3,cat__EquipmentType_Conveyor,0.438797
4,cat__EquipmentType_Compressor,0.416050
5,num__DaysSinceMaintenance,0.347728
6,cat__EquipmentType_Turbine,0.338937
7,cat__Site_Dallas,0.272784
8,num__SensorVibration,0.256212
9,num__SensorTemp,0.248981


## 4) IBM-Style Recommendations

- Run daily batch scoring over active assets.
- Send top-risk alerts into maintenance dispatch tooling.
- Track weekly precision/recall and alert volume against capacity.
- Monitor subgroup performance by equipment type and site.
- Retrain quarterly or when drift and KPI degradation are detected.

## Interview Close

"I built this as a decision pipeline, not just a model: realistic split strategy, threshold governance, and operations-ready alert outputs tied to business KPIs like downtime and maintenance efficiency."